# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodeA7/FlyRank_AI/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This is a provisional Week 1 framing. I can revise the lane through Week 4 as I learn more about the data and the decision. The starter data is anonymized; keep analysis at page level and avoid exposing identifiers.

## 1. My lane (or freestyle) and why

**Refresh / Content Opportunity Scoring (Lane 2).**

I want to help a search and content team decide which existing pages deserve a human review first, rather than assume that every page experiencing a decline needs to be rewritten. The starter dataset contains page-level search and content signals, and my initial findings show that many pages have declining trends despite having some level of search visibility. This makes it worth investigating how these signals can be used to rank pages for review, especially when the team has limited time and resources. However, this is only my starting point, and I may adjust my approach as I learn more over the next few weeks.

## 2. The question: decision, action, cost of a wrong call

**Search question:** Among pages with enough search exposure to assess, which ones should a content team review first for a possible refresh, expansion, protection, or monitoring action?

**Unit of analysis (grain):** one content item/page in the starter CSV (one row per page); pages belong to pseudonymized clients. For later warehouse work, I will explicitly choose a decision date and aggregate at page × decision date if needed.

**Output:** a ranked, reviewable queue with a suggested review reason/action and simple reason codes (for example, observed decline, exposure, freshness, and data sufficiency). Scores prioritize review; they do not prescribe an automatic edit.

**Decision and action:** an SEO/content lead chooses which pages fit the team's review capacity. An editor then checks the page and its context and decides whether to update, expand, protect, monitor, or take no action.

**Cost of error:** a false positive spends scarce editorial/reviewer time and may encourage an unnecessary change; a false negative can leave a genuinely valuable page's problem unreviewed and prolong lost search opportunity. A harmful rewrite could also damage useful content, so the queue must support human review and conservative actions. The relative costs and team capacity are not in this CSV as such I will not pretend to have quantified them yet.

**Why data/ML can help:** there are many pages and several observable signals (visibility, recent movement, freshness, content attributes) that a person cannot consistently prioritize one by one. A transparent baseline can combine these signals, and a model may help rank candidates if it improves top-of-queue usefulness over that baseline on evaluation. The current starter trend label is a contemporaneous proxy, not a future outcome or ground truth that a refresh works. For a stronger capstone, I will investigate leakage-safe future windows in the warehouse release.

## 3. Quick look at the data (real numbers)

Load the starter CSV and compute evidence directly. “Down” is the dataset's existing last-30-days-versus-previous-30-days trend bucket (defined as more than a 20% impressions drop); it is **not** a future prediction. The 100-impression cutoff below is a transparent exploratory volume floor, not a final policy.

In [5]:
import os
from pathlib import Path
import pandas as pd

#cloning into the github repo for the dataset
if not Path("data/raw/content_refresh_anonymized.csv").exists() and not Path("FlyRank_AI").exists():
    print("Cloning repository to fetch dataset...")
    !git clone https://github.com/NodeA7/FlyRank_AI.git

# Works from the repo root, the work/notebooks folder, and Colab after cloning the repo.
candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/FlyRank_AI/data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv")
]
csv_path = next((p for p in candidates if p.exists()), None)

df = pd.read_csv(csv_path)
trend = df["trend_direction"].astype(str).str.lower()
down = trend.eq("down")
visible_down = down & df["impressions_90d"].ge(100)

print(f"Starter dataset: {len(df):,} page rows across {df['client_id'].nunique():,} pseudonymized clients")
print(f"Rows marked down in the current trend comparison: {down.sum():,} / {len(df):,} ({down.mean():.1%})")
print(f"Down rows with >=100 impressions in the 90-day window: {visible_down.sum():,} ({visible_down.mean():.1%} of all rows)")
print(f"Median impressions_90d across all rows: {df['impressions_90d'].median():,.0f}")

Starter dataset: 30,000 page rows across 32 pseudonymized clients
Rows marked down in the current trend comparison: 16,262 / 30,000 (54.2%)
Down rows with >=100 impressions in the 90-day window: 13,152 (43.8% of all rows)
Median impressions_90d across all rows: 731


**What these numbers suggest:** the starter slice contains 30,000 pages across 32 pseudonymized clients; 16,262 (54.2%) are marked down in the current comparison window; and 13,152 (43.8% of all rows) are both down and above the exploratory 100-impression floor. The overall median is 731 impressions over 90 days. That is enough scale and measurable exposure to investigate a ranked review queue, while the 100-impression threshold helps avoid treating every low-volume fluctuation equally. These are descriptive counts from one starter snapshot, not evidence that refreshes cause recovery.

The starter notebook's existing holdout evaluation also reports Precision@50 of 0.740 for its random-forest ranking versus 0.240 for its hand-written baseline on the defined declining proxy. I treat that as a promising starting result from the starter exercise—not an independently reproduced result here, not proof of future performance, and not proof that the recommendations improve outcomes.

## 4. Careful words: what I can and can't claim

I can report descriptive patterns in this anonymized starter slice and, if I define leakage-safe labels and a suitable holdout design, estimate how well a ranking identifies pages meeting that measured label. I can propose a decision-support queue for human review. The current `trend_direction`/`trend_pct` are derived from the same current comparison window; they must not be features when the target is that trend label. The starter file is aggregated over a trailing 90-day window, so it cannot by itself establish a clean feature-before-outcome timeline.

I cannot claim that a refresh caused recovery, that a score guarantees value, that the sample represents all sites, or that I am predicting Google's algorithm. For stronger future-outcome claims I need to define feature and target windows separately in the warehouse, audit leakage and coverage, validate by client and/or time as appropriate, and review top-ranked examples. Any actions remain recommendations for a person to assess.

## Self-check

- [x] Lane, question, grain, output, action, error costs, and role for data/ML are stated.
- [x] The code cell loads the shipped starter CSV and computes real numbers.
- [x] Claims distinguish the current trend proxy from future outcomes and causal impact.
- [x] No client names, URLs, or private queries are included.
- [x] Executed the notebook code cell locally from `work/notebooks/`; it loaded the starter CSV and completed without errors. (Rerun in Colab if that is your submission environment.)
- [ ] Revisit this provisional lane by the end of Week 4.